In [2]:
import numpy as np
import pandas as pd

np.random.seed(42)

n_batches = 100
time_hours = np.arange(0, 121, 1)
data_list = []

for b in range(1, n_batches + 1):
    S = 14.0 + np.random.normal(0, 0.3)  # Initial Sugar (°Plato)
    X = 0.5 + np.random.normal(0, 0.04)  # Initial Yeast (g/L)
    P = 0.0  # Initial Ethanol (% ABV)

    # Base profile setpoint
    base_temp = 12.0 + np.random.uniform(-1.0, 1.2)

    for t in time_hours:
        # Dynamic temperature curve ramp-up
        T = base_temp + (3.5 * (1 / (1 + np.exp(-(t - 36) / 8))))

        # Rate parameters
        T_kelvin = T + 273.15
        mu_max = 0.085 * np.exp(-3200 * (1 / T_kelvin - 1 / 285.15))
        Ks = 1.2
        Y_xs = 0.14
        Y_ps = 0.46

        # Reaction kinetics
        mu = mu_max * (S / (Ks + S))
        dX = mu * X
        dS = -(1 / Y_xs) * dX
        dP = Y_ps * abs(dS)

        # Update states
        S = max(0.0, S + dS)
        X = max(0.0, X + dX)
        P = min(S * Y_ps, P + dP)

        # Exothermic cooling duty calculation (MJ/hr)
        cooling_duty = abs(dS) * 16.5 + (T - 10.0) * 0.75 + np.random.normal(0, 0.2)

        data_list.append(
            {
                "batch_id": f"BATCH_{b:03d}",
                "time_hr": t,
                "fermenter_temp_c": round(T, 2),
                "sugar_plato": round(S, 2),
                "biomass_g_l": round(X, 2),
                "ethanol_abv": round(P, 2),
                "cooling_duty_mj_hr": round(max(0.0, cooling_duty), 2),
            }
        )

df = pd.DataFrame(data_list)
df.to_csv("brewery_fermentation_data.csv", index=False)
print(f"Generated {len(df):,} rows saved to 'brewery_fermentation_data.csv'.")

Generated 12,100 rows saved to 'brewery_fermentation_data.csv'.


In [7]:

from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

# Load dataset
df = pd.read_csv("brewery_fermentation_data.csv")

# Identify actual completion hour (Target variable)
completion_df = (
    df[df["sugar_plato"] <= 2.0]
    .groupby("batch_id")["time_hr"]
    .min()
    .reset_index()
)
completion_df.columns = ["batch_id", "target_completion_hr"]

# Filter early telemetry (Hours 0 to 24)
early_telemetry = df[df["time_hr"] <= 24]

# Feature Engineering
features = (
    early_telemetry.groupby("batch_id")
    .agg(
        avg_temp_24h=("fermenter_temp_c", "mean"),
        max_temp_24h=("fermenter_temp_c", "max"),
        sugar_drop_24h=("sugar_plato", lambda x: x.iloc[0] - x.iloc[-1]),
        biomass_growth_24h=("biomass_g_l", lambda x: x.iloc[-1] - x.iloc[0]),
        cooling_energy_24h=("cooling_duty_mj_hr", "sum"),
    )
    .reset_index()
)

# Merge features with target
dataset = pd.merge(features, completion_df, on="batch_id")

X = dataset.drop(columns=["batch_id", "target_completion_hr"])
y = dataset["target_completion_hr"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model Training
model = XGBRegressor(n_estimators=120, learning_rate=0.04, max_depth=3)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("--- MODEL PERFORMANCE RESULTS ---")
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test, y_pred):.2f} hours")
print(f"R-Squared Score (R2): {r2_score(y_test, y_pred):.3f}")

--- MODEL PERFORMANCE RESULTS ---
Mean Absolute Error (MAE): 0.66 hours
R-Squared Score (R2): 0.140


In [9]:
import numpy as np
from scipy.optimize import minimize


# Cost function: Minimize total energy while penalizing temperature spikes
def cooling_cost_function(temp_stages):
    # temp_stages = [Phase 1, Phase 2, Phase 3, Phase 4]
    # Metabolic heat load factor
    heat_load = np.sum(16.5 * np.exp(0.075 * temp_stages))
    chiller_work = np.sum(0.85 * (temp_stages - 10.0) ** 2)
    return heat_load + chiller_work


# Constraints & Bounds
# Temperatures must stay strictly within 10.0°C and 16.0°C
bounds = [(10.0, 16.0) for _ in range(4)]
initial_temp_schedule = [12.0, 13.5, 15.0, 12.0]

# Optimization solver
res = minimize(cooling_cost_function, initial_temp_schedule, bounds=bounds)

print("--- ENERGY OPTIMIZATION RESULTS ---")
print(f"Initial Unoptimized Energy Load: {cooling_cost_function(np.array(initial_temp_schedule)):.2f} MJ")
print(f"Optimized Energy Load: {res.fun:.2f} MJ")
print(
    f"Energy Reduction: {((cooling_cost_function(np.array(initial_temp_schedule)) - res.fun) / cooling_cost_function(np.array(initial_temp_schedule))) * 100:.1f}%"
)
print("\nOptimal Fermenter Temperature Schedule (°C):")
for idx, temp in enumerate(res.x, 1):
    print(f"  Phase {idx}: {temp:.2f} °C")

--- ENERGY OPTIMIZATION RESULTS ---
Initial Unoptimized Energy Load: 215.87 MJ
Optimized Energy Load: 139.72 MJ
Energy Reduction: 35.3%

Optimal Fermenter Temperature Schedule (°C):
  Phase 1: 10.00 °C
  Phase 2: 10.00 °C
  Phase 3: 10.00 °C
  Phase 4: 10.00 °C
